# Introduction to Quantum Computing
<img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-Summer-QuantumWorkshop/main/images/ionq-quantum.png"
     alt="IONQ-QUANTUM"
     width="50%">
<p>(Finviz, n.d.)

### Prof. David Singletary
### Florida State College at Jacksonville

# Day 5: Amplitude Amplification, Cloud Platforms & Hardware Execution
- Current Events
- Ch. 6: Amplitude Amplification
- Overview of Major Cloud-Based Quantum Platforms
- AWS Braket Execution Workflow
- IBM Quantum
- Assessment
- Wrap-up




## Current Events

<img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-Summer-QuantumWorkshop/main/images/google-doodle-worldquantumday-260414.png"
     alt="World Quantum Day Google Doodle"
     width="25%">

<a href="https://worldquantumday.org/">World Quantum Day (website)</a>

<a href="https://www.youtube.com/watch?v=kVxajebdJqA">World Quantum Day (video)</a>

<a href="https://www.eff.org/deeplinks/2026/04/yikes-encryptions-y2k-moment-coming-years-early">Yikes, Encryption’s Y2K Moment is Coming Years Early</a>

<img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-Summer-QuantumWorkshop/main/images/cryptography-types.png"
     alt="Cryptography Types"
     width="40%">
- (GlassWire, 2024)

<a href="https://www.tomshardware.com/tech-industry/cyber-security/go-maintainer-joins-collective-klaxon-about-encryption-breaking-quantum-computers-developer-urges-immediate-switch-to-post-quantum-methods-to-prevent-worldwide-disaster">Go maintainer joins collective klaxon about encryption-breaking quantum computers — developer urges immediate switch to post-quantum methods to prevent worldwide disaster</a>

<a href="https://bitcoinmagazine.com/news/adam-back-says-quantum-threat-to-bitcoin">Adam Back Says Quantum Threat to Bitcoin Is Decades Away, Urges Gradual Migration to Post-Quantum Security</a>

<a href="https://www.coindesk.com/tech/2026/04/08/attacking-bitcoin-mining-with-a-quantum-computer-would-require-the-energy-of-a-star-academics-say">Attacking bitcoin mining with a quantum computer would require the energy of a star, academics say
</a>


# IONQ Price

In [ ]:
import yfinance as yf
import matplotlib.pyplot as plt

ticker = "IONQ"

# download 10 days of data at 1-hour intervals
data = yf.download(ticker, period="1y", interval="1mo")

# plot the closing price
plt.figure()
plt.plot(data.index, data["Close"])
plt.xlabel("Time")
plt.ylabel("Price")
plt.title(f"{ticker} 1 Year Day Price (1 Month Intervals)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Ch. 6 Amplitude Amplification

### vs. Constructive Interference
- Constructive Interference occurs when amplitudes combine in a way that increases the probability of certain states.
- It is a local effect caused by how gates (like H) recombine amplitudes.
- Amplitude amplification is a systematic process (e.g., in Grover's algorithm) that repeatedly uses interference to boost the probability of good states and suppress others.
- It is a structured algorithm built from **multiple** interference steps.
- In a nutshell, constructive interference is the mechanism, amplitude amplification is the strategy that uses that mechanism repeatedly

### Applying Amplitude Amplification
- Prepare an initial quantum state using a circuit/operator (e.g., "A")
- Construct a **Grover operator** (e.g., "G") which combines an oracle and an **inversion** operator (introduced below)
- Apply G repeatedly (j times) to amplify desired outcomes

  <img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-Summer-QuantumWorkshop/main/images/figure6.3.png"
      alt="Amplitude Amplification Circuit Diagran"
      width="25%">

- Recall our Day 4 examples where we were finding a good outcome in a search space
  - We represented the search space using a uniform superposition over n qubits
  - Each outcome started with equal probability, since we didn't know where the solution we
  - We used a phase oracle to tag the desired outcome(s) by flipping the sign of the amplitudes of the good states
- A phase flip does not change measurement probabilities, so it cannot be observed directly
  - Measurement depends on the magnitude of an amplitude, not its sign (phase)
- We need a way to convert the hidden phase difference into a probability advantage, making the marked state more likely to be measured

## The Magical Mystery Inversion Operator

- This step is often called “inversion by the mean”
- It works by adjusting amplitudes so good outcomes get larger and others get smaller
- Think of it as reflecting (mirroring) the state around an average value (i.e., a mirror placed at the average)
  - Values below the average → reflect upward
  - Values above the average → reflect downward
  - Values exactly on the mirror → stay where they are
- The inversion step flips each value across that average: values below the average go above it, and values above go below it
  - The reflection helps turn the earlier phase change into a higher probability of measuring the correct result
- The reflection formula is
```
        a'=2μ-a

    where

        a = original amplitude
        μ = mean amplitude
        a' = reflected amplitude
```
  - e.g., Suppose the average amplitude is 0.25, and the phase on the gold state has been flipped:

      |State       | Before	Reflection | After Reflection |
      |------------|-------------------|------------------|
      | Good state | -0.25             | +0.75            |
      | Others	   | +0.25             | +0.25            |

- The phase-flipped state starts below the line, so after reflection it jumps farther above it
- The implementation of the inversion operator treats the state as a single vector and uses the dot product* between the original and current states to measure their similarity, then uses that value to reflect the current state back toward the original, enabling amplitude amplification

```
* dot product: a measure of similarity between two vectors,
  computed by multiplying corresponding elements and summing
  the results
```

### Modeling Inversion With Code
- We can model inversion using a Python fuction which computes the inner product between the original state and the phase-flipped (current) state to measure how similar they are
- This similarity value (proj) is used to adjust each amplitude
- Apply the update rule:
```
          current[k] = 2 * proj * original[k] - current[k]
```
- This performs a reflection of the current state toward the original state
- The inner product has magnitude <= 1, ensuring the transformation remains valid and normalized.

## Example: Phase Oracle + Inversion (Amplitude Amplification Step)
- This example demonstrates the two core steps behind amplitude amplification:
  1. Tag a good outcome using a phase oracle
  2. Amplify that outcome using the inversion (diffusion) operator
- Start with a uniform superposition over 3 qubits
- Define a predicate to mark a specific outcome (k == 3)
- Apply a phase oracle to flip the sign of that outcome
- Save the original state as a reference for reflection
- Apply the inversion operator, which reflects the current state around the original state
- Note the magnitude and probability value changes in both the desirable and undesirable outcomes

In [ ]:
# ---- setup for clone/repo imports ----
import os
import sys
import subprocess
import importlib

REPO_URL = "https://github.com/learnqc/code.git"
REPO_DIR = "/content/code"
SRC_DIR = f"{REPO_DIR}/src"

# Install required pip package(s)
subprocess.run(
    ["pip", "install", "-q", "sty"],
    check=True
)

# Clone repo if needed
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

# Make src importable
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Clear any stale imports
importlib.invalidate_caches()

print("Setup complete")

In [ ]:
from math import sqrt
from ch03.util import *
from ch05.sim_circuit import *

# inner product <a|b>
def inner(a, b):
    total = 0
    for k in range(len(a)):
        total += a[k].conjugate() * b[k]
    return total

# classical phase oracle
def oracle(state, predicate):
    for item in range(len(state)):
        if predicate(item):
            state[item] *= -1

# inversion operator
def inversion(original, current):
    proj = inner(original, current)
    for k in range(len(current)):
        current[k] = 2 * proj * original[k] - current[k]

# mark outcome 3 as the good outcome
def predicate(k):
    return k == 3

# create equal superposition over 3 qubits
n = 3
state = [1 / sqrt(2**n) for _ in range(2**n)]

# original state is a reference point needed for the
# reflection so we save it before applying the oracle
s = state.copy()

print("Original state (before oracle)")
print_state_table(state)

# apply phase oracle
oracle(state, predicate)

print("After oracle (good outcome phase-flipped)")
print_state_table(state)

# apply inversion operator
inversion(s, state)

print("After inversion")
print_state_table(state)

## Example: Amplitude Amplification with a Random State
- This example extends the previous one by starting with a random quantum state instead of a uniform superposition, to show that amplitude amplification works more generally, not just in idealized cases.
- Start with a randomly generated, normalized 3-qubit state with unequal amplitudes.
  - Define the predicate (k == 5)
  - Apply a phase oracle
  - Save the original state
  - Apply the inversion operator
- Unlike the uniform case, multiple values shift during inversion
- Again, note the resulting magnitude and probability values of the undesirable outcomes

In [ ]:
import random
# from math import sqrt, atan2, degrees
from ch03.util import *
from ch05.sim_circuit import *

# generate a random normalized quantum state for n qubits
def generate_state(n, seed=None):
    if seed is not None:
        random.seed(seed)

    N = 2**n

    # create random complex amplitudes
    state = [
        complex(random.uniform(-1, 1), random.uniform(-1, 1))
        for _ in range(N)
    ]

    # normalize the state
    norm = sqrt(sum(abs(a)**2 for a in state))
    state = [a / norm for a in state]

    return state

# inner product <a|b>
def inner(a, b):
    total = 0
    for k in range(len(a)):
        total += a[k].conjugate() * b[k]
    return total

# classical phase oracle
def oracle(state, predicate):
    for k in range(len(state)):
        if predicate(k):
            state[k] *= -1

# inversion operator
def inversion(original, current):
    proj = inner(original, current)
    for k in range(len(current)):
        current[k] = 2 * proj * original[k] - current[k]

# good outcome is 5
def predicate(k):
    return k == 5

# create a random 3-qubit state
n = 3
state = generate_state(n)

# save original state before oracle
s = state.copy()

print("Original random state")
print_state_table(state)

# apply phase oracle
oracle(state, predicate)

print("After oracle (outcome 5 phase-flipped)")
print_state_table(state)

# apply inversion
inversion(s, state)

print("After inversion")
print_state_table(state)

## The Grover Iterate: Combining Oracle and Amplification
- A Grover operator (G) combines a quantum oracle (O) and an inversion operator (M)
- It is called an **iterate** because it is applied multiple times in a loop
- Start by preparing an initial state using circuit A
- Apply the Grover operator repeatedly: GʲA (for some number of iterations j)
- Each application of G = O followed by M progressively amplifies the desired outcomes
  <img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-Summer-QuantumWorkshop/main/images/figure6.16.png"
      alt="Grover Iterate"
      width="25%">



Example: Applying Grover Iterations

- This program puts the full amplitude amplification process together by combining a phase oracle and an inversion operator into a Grover iterate.
- Starting from a uniform 3-qubit state, it marks outcome 3 as the good outcome and then applies the Grover operator repeatedly.
- The program shows the state table after 1, 2, and 3 iterations so you can see how the amplitude of the marked state grows.
  - It also checks the expected probability relationships using assert statements.

### Choosing the Right Number of Iterations
- Applying more Grover iterations does not always improve results—too many iterations can reduce the probability of the good outcome
- e.g., for 1 good outcome (n = 3), 2 iterations is optimal, while 3 iterations performs worse
- Amplitude amplification follows an oscillating pattern (increases, peaks, then decreases)
- The optimal number of iterations depends on the number of good outcomes (L)
  - Fewer good outcomes -> more iterations needed
  - More good outcomes -> fewer iterations needed
- The amplitude of good outcomes follows a sinusoidal relationship as iterations increase
- The optimal number of iterations can be approximated by

  - Divide the total number of possible outcomes (2ⁿ) by the number of good outcomes (L)
  - The square root of that value multiplied by π/4 and round down to the nearest whole number
  - i.e.,

  $\left\lfloor \frac{\pi}{4} \sqrt{\frac{2^n}{L}} \right\rfloor$

- Choosing the right number of iterations is critical to maximize the probability of measuring the correct result.
- Grover is powerful, but it assumes you either know or can estimate how many solutions exist; otherwise you have to search for the peak experimentally.

In [ ]:
from math import sqrt, sin, cos, asin
from ch03.util import *
from ch05.sim_circuit import *

# inner product <a|b>
def inner(a, b):
    total = 0
    for k in range(len(a)):
        total += a[k].conjugate() * b[k]
    return total

# classical phase oracle
def oracle(state, predicate):
    for k in range(len(state)):
        if predicate(k):
            state[k] *= -1

# inversion operator
def inversion(original, current):
    proj = inner(original, current)
    for k in range(len(current)):
        current[k] = 2 * proj * original[k] - current[k]

# compare floating-point values
def is_close(a, b, tol=1e-9):
    return abs(a - b) < tol

# classical Grover iteration
def classical_grover(state, predicate, iterations):
    s = state.copy()
    items = [k for k in range(len(state)) if predicate(k)]

    # probability of measuring a good outcome in the initial state
    p = sum(abs(s[k])**2 for k in items)
    theta = asin(sqrt(p))

    # before any Grover iterations, <s|state> should be 1
    assert is_close(inner(s, state), 1)

    for it in range(1, iterations + 1):
        oracle(state, predicate)
        inversion(s, state)

        # check the angle relationship after each Grover iteration
        assert is_close(inner(s, state), cos(2 * it * theta))

        # check the probability of good outcomes after each iteration
        p = sum(abs(state[k])**2 for k in items)
        assert is_close(p, sin((2 * it + 1) * theta)**2)

# expected amplitude for a good outcome in the uniform case
def target_amplitude_uniform(n, l, j):
    theta = asin(sqrt(l / 2**n))
    return sin((2 * j + 1) * theta) / sqrt(l)

# common setup
n = 3
items = [3]
predicate = lambda i: i in items

# --------
# 1 iteration
# --------
state = [1 / sqrt(2**n) for _ in range(2**n)]

print("Initial state (before Grover, j = 0)")
print_state_table(state)

classical_grover(state, predicate, iterations=1)

print("After 1 Grover iteration")
print_state_table(state)

assert is_close(state[items[0]], target_amplitude_uniform(3, 1, 1))

# --------
# 2 iterations
# --------
state = [1 / sqrt(2**n) for _ in range(2**n)]

classical_grover(state, predicate, iterations=2)

print("After 2 Grover iterations")
print_state_table(state)

assert is_close(state[items[0]], target_amplitude_uniform(3, 1, 2))

# --------
# 3 iterations
# --------
state = [1 / sqrt(2**n) for _ in range(2**n)]

classical_grover(state, predicate, iterations=3)

print("After 3 Grover iterations")
print_state_table(state)

assert is_close(state[items[0]], target_amplitude_uniform(3, 1, 3))

### OO Version
- Here's a version of the above program which uses the QuantumRegister and QuantumCircuit classes along with qc.run() to generate the initial uniform superposition.
- This better aligns with how quantum states are actually prepared in a circuit, using gates rather than manually constructing vectors.

In [ ]:
from math import sqrt, sin, cos, asin
from ch03.util import *
from ch05.sim_circuit import *

# inner product <a|b>
def inner(a, b):
    total = 0
    for k in range(len(a)):
        total += a[k].conjugate() * b[k]
    return total

# phase oracle
def oracle(state, predicate):
    for k in range(len(state)):
        if predicate(k):
            state[k] *= -1

# inversion operator
def inversion(original, current):
    proj = inner(original, current)
    for k in range(len(current)):
        current[k] = 2 * proj * original[k] - current[k]

# compare floating-point values
def is_close(a, b, tol=1e-9):
    return abs(a - b) < tol

# Grover iteration
def grover(state, predicate, iterations):
    s = state.copy()
    items = [k for k in range(len(state)) if predicate(k)]

    # probability of measuring a good outcome in the initial state
    p = sum(abs(s[k])**2 for k in items)
    theta = asin(sqrt(p))

    # before any Grover iterations, <s|state> should be 1
    assert is_close(inner(s, state), 1)

    for it in range(1, iterations + 1):
        oracle(state, predicate)
        inversion(s, state)

        # check the angle relationship after each Grover iteration
        assert is_close(inner(s, state), cos(2 * it * theta))

        # check the probability of good outcomes after each iteration
        p = sum(abs(state[k])**2 for k in items)
        assert is_close(p, sin((2 * it + 1) * theta)**2)

# expected amplitude for a good outcome in the uniform case
def target_amplitude_uniform(n, l, j):
    theta = asin(sqrt(l / 2**n))
    return sin((2 * j + 1) * theta) / sqrt(l)

# predicate function
def is_target(i):
    return i in items

# common setup
n = 3
items = [3]
predicate = is_target

# Step 1: create a 3-qubit circuit
q = QuantumRegister(n)
qc = QuantumCircuit(q)

# Step 2: apply H to all qubits for a uniform superposition
for i in range(n):
    qc.h(q[i])

# Step 3: use qc.run() to get the initial state
initial_state = qc.run()

# --------
# 1 iteration
# --------
state = initial_state.copy()

print("Initial state (before Grover, j = 0)")
print_state_table(state)

grover(state, predicate, iterations=1)

print("After 1 Grover iteration")
print_state_table(state)

assert is_close(state[items[0]], target_amplitude_uniform(3, 1, 1))

# --------
# 2 iterations
# --------
state = initial_state.copy()

grover(state, predicate, iterations=2)

print("After 2 Grover iterations")
print_state_table(state)

assert is_close(state[items[0]], target_amplitude_uniform(3, 1, 2))

# --------
# 3 iterations
# --------
state = initial_state.copy()

grover(state, predicate, iterations=3)

print("After 3 Grover iterations")
print_state_table(state)

assert is_close(state[items[0]], target_amplitude_uniform(3, 1, 3))

## Overview of Major Cloud-Based Quantum Platforms
Quantum computing resources are accessible remotely through cloud services rather than requiring local hardware.
- Leading providers such as IBM (IBM Quantum), Microsoft (Azure Quantum), Amazon (Amazon Braket), IonQ, and Google offer web-based interfaces, software development kits (SDKs), simulators, and access to real quantum processors.
- These platforms allow users to design and test circuits on classical simulators before submitting them to physical quantum hardware, where real-world effects such as noise and decoherence become apparent.
- Simulators enable rapid development and debugging in a controlled environment, while real devices provide practical insight into hardware limitations and performance.
- By integrating both resources within cloud infrastructure, these platforms support scalable hybrid workflows and make quantum computing broadly accessible to researchers, developers, and students worldwide.



## AWS Braket
- Braket is a cloud service for building, testing, and running quantum computing programs
- It provides access to both simulators and real quantum hardware from multiple vendors (e.g., IonQ, Rigetti, OQC)
- Users can design circuits using the Braket SDK and execute them locally or as managed cloud tasks
- It supports a hybrid workflow, combining classical code with quantum circuits in the same program
- Braket handles infrastructure, so you can focus on algorithms instead of hardware setup and maintenance

- AWS provides a basic digital badge path which introduces the SDK.
  - It also provides more advanced learning paths

  https://aws.amazon.com/blogs/quantum-computing/introducing-the-amazon-braket-learning-plan-and-digital-badge/

## Running a Quantum Circuit with AWS Braket (Local Simulator)
- In this example, we use the AWS Braket SDK to build and run a simple quantum circuit.
- Instead of submitting a job to AWS cloud hardware, we use a local simulator, which runs entirely in the Colab environment.
- This allows us to experiment with quantum circuits quickly without needing AWS credentials or incurring any cost.
- The circuit below creates a Bell state, one of the examples of quantum entanglement that we covered previously.
  - Apply a Hadamard (H) gate to qubit 0 to create a superposition
  - Apply a CNOT gate to entangle qubit 0 and qubit 1
- After running the circuit multiple times (shots), we examine the measurement results. For a Bell state, we expect to see only `00` and `11` with roughly equal probability.

In [ ]:
!pip -q install amazon-braket-sdk

from braket.circuits import Circuit
from braket.devices import LocalSimulator

# create a Bell-state circuit
circuit = Circuit()
circuit.h(0)
circuit.cnot(0, 1)

# run on Braket local simulator
device = LocalSimulator()
task = device.run(circuit, shots=1000)
result = task.result()

print("Circuit:")
print(circuit)

print("\nMeasurement counts:")
print(result.measurement_counts)

## IBM Quantum

https://quantum.cloud.ibm.com/learning/en

- Access to:
  - Simulators (fast, no queue)
  - Real quantum computers (slower, but real hardware)



## Estimators and Samplers
- Qiskit provides runtime primitives:
  - **Estimator**
    - Compute expectation values of observables (e.g., ⟨Z⟩, ⟨XX⟩)
    - Input: circuit + observables
    - Output: numbers (averages), not raw measurement counts
  - Useful for:
    - algorithms (e.g., VQE)
    - analyzing quantum states
    - measuring correlations and properties
    - Often includes error mitigation automatically
  - **Sampler**
    - Perform shot-based sampling (like real measurements)
    - Input: circuit(s)
    - Output: counts or probability distributions (e.g., {00: 512, 11: 488})
  - Useful for:
      - simulating actual experiment outcomes
      - debugging circuits
      - teaching measurement behavior

https://quantum.cloud.ibm.com/

### Create an Instance (first time only)
<img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-Summer-QuantumWorkshop/main/images/ibm-quantum1.png"
     alt="IBM Quantum: Create Instance"
     width="50%">

<img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-Summer-QuantumWorkshop/main/images/ibm-quantum2.png"
     alt="IBM Quantum: Instance Created"
     width="50%">
     
<img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-Summer-QuantumWorkshop/main/images/ibm-quantum3.png"
     alt="IBM Quantum: Running Hello World"
     width="50%">

## Create API Key
<img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-Summer-QuantumWorkshop/main/images/ibm-quantum4.png"
     alt="IBM Quantum: Create an API Key"
     width="50%">

## Using Secrets in Google Colab

### Don't Hardcode Your Keys!
- e.g., api_key = "hardcoded_key"
- Hardcoding exposes sensitive data
  - Risky if notebook is shared or uploaded
- Best Practice
  - Use Secrets for all API keys and tokens
  - Keep notebooks clean and shareable without exposing credentials
- Open the Secrets panel from the left sidebar in Colab (the "key" icon)

  <img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-Summer-QuantumWorkshop/main/images/colab-key-icon.png"
      alt="Colab Key Button"
      width="10%">

- Click “Add a new secret”
- Enter a name (e.g., IBM_QUANTUM_KEY)
- Enter the value (your actual key)
- Secrets are stored securely and not shown in the notebook cells
### Accessing a Secret in Code
```
from google.colab import userdata

api_key = userdata.get("IBM_API_KEY")
```
- Use the returned value just like any variable
- Secrets are not visible to others unless they have access to your environment
- They are persistent across sessions (in your account)
- If a secret changes, update it in the Secrets panel, not in code


## Create an Instance (for subsequent executions)

https://quantum.cloud.ibm.com/instances

<img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-Summer-QuantumWorkshop/main/images/ibm-quantum-create-instance.png"
     alt="IBM Quantum: Create an Instance"
     width="75%">

- This is where you can obtain a CRN (Cloud Resource Name)

## Setup: Install Qiskit Runtime Library

In [ ]:
!pip install qiskit-ibm-runtime
!pip install pylatexenc

## Example Program

- The following code blocks provide an example of running on actual IBM hardware from this notebook.
  - The runtime setup block requires your personal API KEY using the Colab secrets feature and the CRN from your instance

### Set Up Runtime with Token

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Use the 44-character API_KEY from the IBM Quantum Platform Home dashboard
# CRN (cloud resource name)/instance string are obtained by adding a reesource
# (account/instances/resources/quantum service)
from google.colab import userdata

api_key = userdata.get('IBM_QUANTUM_KEY')
crn = "your_crn"
QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token=api_key,
    instance=crn,
    overwrite=True,
    set_as_default=True
)
service = QiskitRuntimeService()

### Run the Program
- Build a simple 2-qubit quantum circuit that creates entanglement using a Hadamard (H) and controlled-X (CX) gate
- Define a set of Pauli observables (e.g., IZ, XX) to measure properties of the quantum state rather than raw bit outcomes
  - A Pauli observable is a measurement operator that can act on one or more qubits
  - IZ measures qubit 1 in the Z (standard 0/1) basis while leaving qubit 0 unchanged
  - XX measures both qubits in the X (superposition) basis
  - These allow observation of correlations that aren’t visible in standard measurements
- Connect to IBM Quantum and select the least busy real quantum backend (not a simulator)
- Transpile the circuit to match the chosen hardware and map observables to the device layout
- Execute the circuit using the Estimator primitive to compute expectation values on real hardware and return a job ID for tracking results

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorOptions
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from matplotlib import pyplot as plt
import pylatexenc
# Uncomment the next line if you want to use a simulator:
# from qiskit_ibm_runtime.fake_provider import FakeBelemV2

# Create a new circuit with two qubits
qc = QuantumCircuit(2)

# Add a Hadamard gate to qubit 0
qc.h(0)

# Perform a controlled-X gate on qubit 1, controlled by qubit 0
qc.cx(0, 1)

# Return a drawing of the circuit using MatPlotLib ("mpl").
# These guides are written by using Jupyter notebooks, which
# display the output of the last line of each cell.
# If you're running this in a script, use `print(qc.draw())` to
# print a text drawing.
print(qc.draw())

# Set up six different observables.

observables_labels = ["IZ", "IX", "ZI", "XI", "ZZ", "XX"]
observables = [SparsePauliOp(label) for label in observables_labels]

service = QiskitRuntimeService()

backend = service.least_busy(simulator=False, operational=True)

# Convert to an ISA circuit and layout-mapped observables.
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# DS isa_circuit.draw("mpl", idle_wires=False)

# Construct the Estimator instance.

estimator = Estimator(mode=backend)
estimator.options.resilience_level = 1
estimator.options.default_shots = 5000

mapped_observables = [
    observable.apply_layout(isa_circuit.layout) for observable in observables
]

# One pub, with one circuit to run against five different observables.
job = estimator.run([(isa_circuit, mapped_observables)])

# Use the job ID to retrieve your job data later
print(f">>> Job ID: {job.job_id()}")
print(job.backend())

## Saved Output of Above Block
```
     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘
>>> Job ID: d7d6dch5a5qc73dpggk0
<IBMBackend('ibm_fez')>
```

### Show the results
- A "Pub" is IBM terminology for "primitive unified block", a unit of work sent to a primitive like Estimator or Sampler.
- This code displays one Pub (one circuit + a list of observables),

- This can take several minutes, depending on resource availability

In [ ]:
# This is the result of the entire submission.  You submitted one Pub,
# so this contains one inner result (and some metadata of its own).
job_result = job.result()

# This is the result from our single pub, which had six observables,
# so contains information on all six.
pub_result = job.result()[0]

for label, value in zip(observables_labels, pub_result.data.evs):
    print(f"{label}: {value}")

## Saved Output of Above Block
```
IZ: 0.0016330227825815161
IX: 0.015921972130169777
ZI: -0.00391729308926761
XI: 0.023939013323302075
ZZ: 0.946301872886216
XX: 1.0092695765563229
```

### Plot the Result

In [ ]:
# Plot the result

values = pub_result.data.evs

errors = pub_result.data.stds

# plotting graph
plt.plot(observables_labels, values, "-o")
plt.xlabel("Observables")
plt.ylabel("Values")
plt.show()

## Saved Output of Above Block

<img src="https://raw.githubusercontent.com/FSCJ-FacultyDev/WC26-Summer-QuantumWorkshop/main/images/ibm-quantum-example-plot.png"
     alt="IBM Quantum Example Plot"
     width="50%">



### How to Interpret These Results
- Near zero values
  - IZ, IX, ZI, XI ≈ 0
  - Each qubit by itself looks random.
    - No strong individual bias
  - Measuring one qubit alone gives ~50/50
- Large positive values
  - ZZ ≈ 0.95
  - XX ≈ 0.92
  - Together, they are strongly correlated (entangled).
    - In the Z basis → they match (00 or 11)
    - In the X basis → they also match
  - This is the signature of an entangled Bell state.
- For this circuit:
  - H-> CX creates (|00⟩ + |11⟩) / √2
  - The results confirm ndividually random but jointly correlated values
- It isn't exactly 1.0 because real hardware isn't perfect.
- The values can be slightly different from 1 due to noise, gate errors, or measurement error

# Others

## Microsoft Azure Quantum

https://learn.microsoft.com/en-us/training/paths/quantum-computing-fundamentals/

Google Quantum

https://quantumai.google/resources

## IonQ

https://www.ionq.com/explainers/lecture-series-introduction-to-quantum-computers


# Assessment



## References

- Gonciulea, C., & Stefanski, C. (2025). Building Quantum Software With Python: A Developer’s Guide. Manning Publications. ISBN: 978-1633437630.
- GlassWire. (2024, February 8). Cybersecurity horizons 2024: Navigating the next wave of digital defense. https://www.glasswire.com/blog/2024/02/08/cybersecurity-horizons-2024-navigating-the-next-wave-of-digital-defense/